In [33]:
# Enable autoreload when modifying files
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [34]:
# Show current path
from pathlib import Path
print(f"Current path: {Path.cwd()}")

Current path: /app/notebooks


In [35]:
# Convert existing csvs to parquet
from icare_risk.utils import convert_csvs_to_parquet
path = '/app/data/synthetic/2026-09-04_234319'
path = '/app/data/synthetic/2026-07-30_151244'
convert_csvs_to_parquet(directory_path=path, delete_originals=False)

Converting: icare_episodes_anon.csv -> icare_episodes_anon.parquet
Converting: icare_microbiology_anon.csv -> icare_microbiology_anon.parquet
Converting: icare_pathology_blood_anon.csv -> icare_pathology_blood_anon.parquet
Converting: icare_pharmacy_prescribing_anon.csv -> icare_pharmacy_prescribing_anon.parquet
Converting: icare_problems_anon.csv -> icare_problems_anon.parquet
Converting: icare_vital_signs_anon.csv -> icare_vital_signs_anon.parquet
Successfully converted 6 files.


In [36]:
def build_feature_matrix(episodes_path: str, table_paths: dict) -> pd.DataFrame:
    # 1. Initialize the dataset engine
    dataset = ClinicalDataset(episodes_path, table_paths)
    
    # 2. Compute requested phenotypes
    fever_df = compute_fever(dataset)
    
    # 3. Build the final matrix
    # Assuming episodes.parquet is the base table we want to enrich
    base_df = pd.read_parquet(episodes_path)[['subject', 'admission_time']]
    
    # Merge features
    final_df = pd.merge(base_df, fever_df, on=['subject', 'admission_time'], how='left')
    final_df = pd.merge(final_df, esbl_df, on=['subject', 'admission_time'], how='left')
    
    # Fill missing with False (meaning no evidence of phenotype)
    final_df.fillna(False, inplace=True)
    
    return final_df

In [37]:
# -----------------------------------------------------
# Configuration
# -----------------------------------------------------
from icare_risk.core.dataset import ClinicalDataset

# Paths to your downloaded Snowflake data
episodes_path = f'{path}/icare_episodes_anon.parquet'
table_paths = {
    'vitals': f'{path}/icare_vital_signs_anon.parquet',
    'pathology': f'{path}/icare_pathology_blood_anon.parquet',
    'microbiology': f'{path}/icare_microbiology_anon.parquet',
    'problems': f'{path}/icare_problems_anon.parquet'
}

# Map the unique column names from the episodes table
episode_config = {
    'subject_col': 'SUBJECT',
    'admission_date': 'ADMISSION_DATE',
    'admission_time': 'ADMISSION_TIME',
    'admission_col': 'std_admission_time',
    'discharge_date': 'DISCHARGE_DATE',

}

table_config = {
    'vitals': {
        'time_column': 'OBSERVATION_PERFORMED_DT',
        'name_column': 'OBSERVATION_NAME',
        'code_column': 'OBSERVATION_CODE',
        'value_column': 'OBSERVATION_RESULT_CLEAN'
    },
    'pathology': {
        'time_column': 'SAMPLE_COLLECTED_DT',
        'name_column': 'ORDER_NAME',
        'code_column': 'ORDER_CODE',
        'value_column': 'RESULT_CLEANED'
    },
    'microbiology': {
        'time_column': 'LATEST_COLLECT_DT',
        'name_column': 'TEST_NAME',
        'code_column': 'TEST_CODE',
        'value_column': 'ORGANISM_BUG' # Or whatever holds the result
    },
    'problems': {
        'time_column': 'PROBLEM_DT_TM',
        'name_column': 'PROBLEM_DESC',
        'code_column': 'PROBLEM_CODE',
        'value_column': None # Problems might not have a numeric value
    }
}

# Initialize the engine
dataset = ClinicalDataset(
    episodes_path=episodes_path,
    episode_config=episode_config,
    table_paths=table_paths,
    table_config=table_config
)

In [38]:
# Visualise a patients summary

from icare_risk.core.inspect import print_patient_timeline
from icare_risk.core.inspect import print_patient_multitable_timeline

print_patient_timeline(dataset, 10001, table_name='vitals')
print_patient_multitable_timeline(dataset, 10001,
    table_names=['vitals', 'pathology', 'microbiology', 'problems']
)

=== TIMELINE FOR PATIENT 10001 | TABLE: VITALS ===

Admission: 2021-09-20 04:00:00  --->  Discharge: 2021-09-26 23:59:59
  [HISTORICAL] 5    events. First: 2021-09-20 00:00:00 | Last: 2021-09-20 00:00:00
  [CURRENT]    219  events. First: 2021-09-20 04:00:00 | Last: 2021-09-26 20:00:00

=== MULTI-TABLE TIMELINE FOR PATIENT 10001 ===

[STAY] 2021-09-20 04:00:00 to 2021-09-26 23:59:59
  > VITALS:
      Historical: 5    records (First: 2021-09-20 00:00:00 | Last: 2021-09-20 00:00:00)
      Current:    219  records (First: 2021-09-20 04:00:00 | Last: 2021-09-26 20:00:00)
  > PATHOLOGY:
      Historical: 1    records (First: 2021-09-20 00:00:00 | Last: 2021-09-20 00:00:00)
      Current:    29   records (First: 2021-09-20 20:00:00 | Last: 2021-09-26 20:00:00)
  > MICROBIOLOGY:
      Historical: 1    records (First: 2020-07-22 00:00:00 | Last: 2020-07-22 00:00:00)
      Current:    0    records
  > PROBLEMS:
      Historical: 0    records
      Current:    0    records



In [40]:
# -------------------------------------------------
# Phenotype examples
# -------------------------------------------------
import pandas as pd

from icare_risk.core.inspect import print_df
from icare_risk.phenotypes.sample import compute_fever
from icare_risk.phenotypes.sample import compute_hypoxia
from icare_risk.phenotypes.sample import compute_test_phenotype

# Compute fever
df_fever = compute_fever(dataset)
print_df(df_fever)

# Compute hypoxia
df_hypoxia = compute_hypoxia(dataset)
print_df(df_hypoxia)

# Compute test phenotype
df_testphn = compute_test_phenotype(dataset)
print_df(df_testphn)



=== Shape: (200, 3) | True in 'is_fever': 194 ===
   SUBJECT  std_admission_time  is_fever
0    10001 2021-09-20 04:00:00      True
1    10002 2020-11-04 12:53:00      True
2    10003 2021-05-06 03:58:00      True
3    10004 2021-01-05 13:52:00      True
4    10005 2023-11-11 04:46:00      True

=== Shape: (200, 3) | True in 'is_hypoxic': 200 ===
   SUBJECT  std_admission_time  is_hypoxic
0    10001 2021-09-20 04:00:00        True
1    10002 2020-11-04 12:53:00        True
2    10003 2021-05-06 03:58:00        True
3    10004 2021-01-05 13:52:00        True
4    10005 2023-11-11 04:46:00        True
(54196, 11) ['SUBJECT', 'ENCNTR_ID', 'OBSERVATION_CODE', 'OBSERVATION_NAME', 'OBSERVATION_PERFORMED_DT', 'OBSERVATION_START_DT', 'OBSERVATION_END_DT', 'OBSERVATION_RESULT_CLEAN', 'OBSERVATION_UNIT', 'standard_time', 'std_admission_time']
(7411, 15) ['SUBJECT', 'PBAID', 'LABORATORY_DEPARTMENT', 'ORDER_CODE', 'ORDER_NAME', 'RESULT_CLEANED', 'RESULT_LOWER_RANGE', 'RESULT_UPPER_RANGE', 'SAMPLE

In [29]:
from icare_risk.phenotypes.pitt import compute_pitt_fever_score
from icare_risk.phenotypes.pitt import compute_pitt_hypo_score
from icare_risk.phenotypes.sirs import derive_sirs_tachycardia
from icare_risk.phenotypes.sirs import derive_sirs_tachypnea
from icare_risk.phenotypes.sirs import derive_sirs_abnormal_temp
from icare_risk.phenotypes.sirs import derive_sirs_abnormal_wbc

df1 = compute_pitt_fever_score(dataset)
df2 = compute_pitt_hypo_score(dataset)

df3 = derive_sirs_tachycardia(dataset)
df4 = derive_sirs_tachypnea(dataset)
df5 = derive_sirs_abnormal_temp(dataset)
df6 = derive_sirs_abnormal_wbc(dataset)

print_df(df1)
print_df(df2)
print_df(df3)
print_df(df4)
print_df(df5)
print_df(df6)

[autoreload of icare_risk.phenotypes.pitt failed: Traceback (most recent call last):
  File "/usr/local/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 276, in check
    superreload(m, reload, self.old_objects)
  File "/usr/local/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 500, in superreload
    update_generic(old_obj, new_obj)
  File "/usr/local/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 397, in update_generic
    update(a, b)
  File "/usr/local/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 309, in update_function
    setattr(old, name, getattr(new, name))
ValueError: compute_pitt_fever_score() requires a code object with 0 free vars, not 5
]


AttributeError: 'MockClinicalDataset' object has no attribute 'con'

In [26]:
import pandas as pd

# Libraries
from icare_risk.core.builder import build_feature_matrix
from icare_risk.core.dataset import MockClinicalDataset
from icare_risk.core.utils import load_phenotypes_config

# Define .yaml path
yaml_path = '/app/src/icare_risk/config/phenotypes_syn.yaml'

# Load phenotypes for sirs
sirs_phenotypes_config = load_phenotypes_config(
    yaml_path=yaml_path, prefix='sirs'
)
print(sirs_phenotypes_config.keys())

# Create mock dataset
dataset = MockClinicalDataset()

# Compute feature matrix
final_features = build_feature_matrix(dataset, sirs_phenotypes_config)

print("\n--- SUM ---")
print(final_features.sum(axis=0))
print("\n--- FINAL FEATURE MATRIX ---")
print(final_features)

dict_keys(['sirs_tachycardia_flag', 'sirs_tachypnea_flag', 'sirs_abnormal_temp_flag', 'sirs_abnormal_wbc_flag'])
Computing: sirs_tachycardia_flag...
Computing: sirs_tachypnea_flag...
Computing: sirs_abnormal_temp_flag...
Computing: sirs_abnormal_wbc_flag...

--- SUM ---
sirs_tachycardia_flag      2
sirs_tachypnea_flag        2
sirs_abnormal_temp_flag    3
sirs_abnormal_wbc_flag     3
dtype: int64

--- FINAL FEATURE MATRIX ---
                            sirs_tachycardia_flag  sirs_tachypnea_flag  \
SUBJECT std_admission_time                                               
101     2026-09-01                              1                    1   
102     2026-09-02                              0                    0   
103     2026-09-03                              0                    0   
104     2026-09-04                              0                    1   
105     2026-09-05                              1                    0   

                            sirs_abnormal_temp_flag

In [22]:
def get_expected_sirs_output():
    """
    Returns the exact expected feature matrix for the MockClinicalDataset.
    """
    expected_df = pd.DataFrame([
        # [SUBJECT, std_admission_time, Tachycardia, Tachypnea, Temp, WBC, Total]
        [101, '2026-09-01', 1, 1, 1, 1, 4],  # Classic Sepsis
        [102, '2026-09-02', 0, 0, 0, 0, 0],  # Healthy
        [103, '2026-09-03', 0, 0, 1, 1, 2],  # Hypothermic & Leukopenic
        [104, '2026-09-04', 0, 1, 0, 1, 2],  # PaCO2 & Bands triggered
        [105, '2026-09-05', 1, 0, 1, 0, 2]   # Missing labs, defaults to 0
    ], columns=[
        'SUBJECT', 
        'std_admission_time', 
        'sirs_tachycardia_flag', 
        'sirs_tachypnea_flag', 
        'sirs_abnormal_temp_flag', 
        'sirs_abnormal_wbc_flag',
        'total_sirs_score'
    ])
    
    # Set the index to match the pipeline output
    expected_df = expected_df.set_index(['SUBJECT', 'std_admission_time'])
    
    return expected_df

# View the expected output
expected_output = get_expected_sirs_output()
print(expected_output)

                            sirs_tachycardia_flag  sirs_tachypnea_flag  \
SUBJECT std_admission_time                                               
101     2026-09-01                              1                    1   
102     2026-09-02                              0                    0   
103     2026-09-03                              0                    0   
104     2026-09-04                              0                    1   
105     2026-09-05                              1                    0   

                            sirs_abnormal_temp_flag  sirs_abnormal_wbc_flag  \
SUBJECT std_admission_time                                                    
101     2026-09-01                                1                       1   
102     2026-09-02                                0                       0   
103     2026-09-03                                1                       1   
104     2026-09-04                                0                       1   
105    

In [41]:
# -------------------------------------------------------------------------
# EXAMPLE 2. DERIVE HISTORICAL CONDITION
# -------------------------------------------------------------------------
# Load phenotypes for sirs
charlson_phenotypes_config = load_phenotypes_config(
    yaml_path=yaml_path, prefix='hx_'
)
print(charlson_phenotypes_config.keys())

# ------------------------------
# Main
# ------------------------------

final_features = build_feature_matrix(dataset, charlson_phenotypes_config)

print("\n--- FINAL FEATURE MATRIX ---")
print(final_features)

[autoreload of icare_risk.phenotypes.pitt failed: Traceback (most recent call last):
  File "/usr/local/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 276, in check
    superreload(m, reload, self.old_objects)
  File "/usr/local/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 500, in superreload
    update_generic(old_obj, new_obj)
  File "/usr/local/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 397, in update_generic
    update(a, b)
  File "/usr/local/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 309, in update_function
    setattr(old, name, getattr(new, name))
ValueError: compute_pitt_fever_score() requires a code object with 5 free vars, not 0
]


dict_keys(['hx_mi_flag', 'hx_copd_flag'])


TypeError: ClinicalDataset.__init__() missing 2 required positional arguments: 'table_paths' and 'table_config'

In [28]:
# ------------------------------------------------------------------------
# Increment ESBL
# ------------------------------------------------------------------------
# Load phenotypes for sirs
increment_phenotypes_config = load_phenotypes_config(
    yaml_path=yaml_path, prefix='bsi_'
)
print(increment_phenotypes_config.keys())

final_features = build_feature_matrix(dataset, increment_phenotypes_config)

print("\n--- FINAL FEATURE MATRIX ---")
print(final_features)

dict_keys(['bsi_not_urinary_flag', 'bsi_non_ecoli_flag'])
Computing: bsi_not_urinary_flag...


AttributeError: module '__main__' has no attribute 'derive_bsi_not_urinary'

In [10]:
from icare_risk.phenotypes.pitt import compute_pitt_score
df_pitt = compute_pitt_score(dataset)
print(pitt)

['Oxygen Saturation' 'Body Temperature' 'Diastolic Blood Pressure'
 'Mean Arterial Pressure' 'Systolic Blood Pressure' 'Heart Rate'
 'Respiratory Rate']


CatalogException: Catalog Error: Table with name interventions does not exist!
Did you mean "pg_indexes"?